In [ ]:
import databento as db
import pandas as pd
import os
import sys
import time
import gc # Garbage Collector pour gérer la RAM

# ==========================================
# 1. PARAMÈTRES UTILISATEUR
# ==========================================
# 👇 COLLEZ VOTRE CLÉ API ICI
API_KEY = "db-tfFsTGSmRYUHjWGVrUshsYd4hgseg" 

# 👇 NOM EXACT DU SSD
NOM_DU_SSD = "Extreme Pro"

# Chemin de sauvegarde
BASE_DIR = f"/Volumes/{NOM_DU_SSD}/Data_DePrado"

# ==========================================
# 2. VÉRIFICATION DU MATÉRIEL
# ==========================================
if not os.path.exists(f"/Volumes/{NOM_DU_SSD}"):
    raise FileNotFoundError(f"❌ LE DISQUE EST ABSENT !\nLe disque '{NOM_DU_SSD}' n'est pas détecté dans /Volumes.")

# Création du dossier
os.makedirs(BASE_DIR, exist_ok=True)

print(f"✅ SYSTÈME PRÊT.")
print(f"📂 Destination : {BASE_DIR}")
print(f"💻 Machine : Mac M3 détecté (Compression ZSTD activée)")

# Connexion API
client = db.Historical(API_KEY)

In [ ]:
# ==========================================
# LE PLAN DE BATAILLE : 6 PÉRIODES CLÉS
# Coût total estimé : ~$105.00
# ==========================================
periods = [
    # 1. Baseline Calme (Pour tester le bruit)
    {"name": "01_Calm_Aug2017", "start": "2017-08-01", "end": "2017-08-31"},
    
    # 2. Flash Crash Volmageddon (Pour tester les stops/labeling)
    {"name": "02_Volmageddon_Feb2018", "start": "2018-02-01", "end": "2018-02-28"},
    
    # 3. Krach COVID (Pour tester la microstructure/Dollar Bars)
    {"name": "03_Crash_Mar2020", "start": "2020-03-01", "end": "2020-03-31"},
    
    # 4. Top 2022 (Pour tester le changement de régime Bull->Bear)
    {"name": "04_Top_Jan2022", "start": "2022-01-01", "end": "2022-01-31"},
    
    # 5. Bottom 2022 (Pour tester le Bet Sizing dans l'indécision)
    {"name": "05_Bottom_Oct2022", "start": "2022-10-01", "end": "2022-10-31"},

    # 6. Crise SVB 2023 (Pour tester le Feature Importance/Découplage)
    {"name": "06_SVB_Crisis_Mar2023", "start": "2023-03-01", "end": "2023-03-31"}
]

print(f"🎯 Liste chargée : {len(periods)} périodes prêtes à être téléchargées.")

In [ ]:
import gc # On s'assure que le Garbage Collector est importé

print("🚀 DÉMARRAGE DU TÉLÉCHARGEMENT SÉQUENTIEL (CORRIGÉ)...")
print("Ne fermez pas VS Code.\n")

total_files = len(periods)

for i, p in enumerate(periods, 1):
    file_path = f"{BASE_DIR}/ES_{p['name']}.parquet"
    
    print(f"🔹 [{i}/{total_files}] Période : {p['name']}")
    
    # 1. Vérification existence
    if os.path.exists(file_path):
        try:
            size_gb = os.path.getsize(file_path) / (1024**3)
            print(f"   ⏩ Déjà téléchargé ({size_gb:.2f} GB). Passage au suivant.")
            continue
        except OSError:
            print("   ⚠️ Fichier existant corrompu ou illisible. On retélécharge.")

    try:
        # 2. Téléchargement API
        # Note : L'avertissement "> 5GB" est normal, on l'ignore car on traite fichier par fichier
        print(f"   📥 Téléchargement en cours ({p['start']} -> {p['end']})...")
        data = client.timeseries.get_range(
            dataset='GLBX.MDP3',
            symbols=['ES.c.0'],
            schema='mbp-1',
            start=p['start'],
            end=p['end'],
            stype_in='continuous'
        )
        
        # 3. Conversion RAM
        print("   ⚙️  Conversion en DataFrame...")
        
        # --- CORRECTION ICI : .to_df() au lieu de .to_dataframe() ---
        df = data.to_df() 
        
        rows = len(df)
        
        # 4. Sauvegarde SSD (Compression ZSTD)
        print(f"   💾 Écriture sur SSD ({rows:,} lignes)...")
        df.to_parquet(file_path, compression='zstd')
        
        # 5. Nettoyage Mémoire (CRUCIAL pour 16GB RAM)
        del df
        del data
        gc.collect() # On force le nettoyage de la RAM immédiatement
        
        print("   🎉 Succès !")
        
        # Pause de courtoisie pour l'API
        time.sleep(2)
        
    except Exception as e:
        print(f"   ❌ ERREUR CRITIQUE : {e}")
        if "memory" in str(e).lower():
            print("🛑 STOP : Mémoire saturée. Redémarrez le Kernel.")
            break

print("\n🏁 TÉLÉCHARGEMENT TERMINÉ.")
print(f"Vérifiez votre dossier : {BASE_DIR}")